# Cross-Domain Analysis: MallWorld Dreams × NDE Phenomenology

## Research Question

Do MallWorld collective dreams and Near-Death Experiences represent the **same realm** (World of Spirits) serving **different purposes**?

### Initial Hypothesis (revised after analysis)

~~Original hypothesis was "different altitudes" — MallWorld lower, NDE higher.~~

### Revised Hypothesis

Both MallWorld and NDE represent the **World of Spirits**, but serve different **uses**:

| Dataset | Use/Purpose | Function |
|---------|-------------|----------|
| **NDE** | Reception/Return | Soul receives exactly what's needed to return to physical life |
| **MallWorld** | Digestion/Vastation | Correspondential processing of spiritual states |

### What This Explains

| Aspect | MallWorld | NDE | Why the Difference |
|--------|-----------|-----|-------------------|
| Atmosphere | 64% negative | 48% positive | **Purpose**: Processing requires confrontation; return requires encouragement |
| Entity character | Mixed (threats) | Overwhelmingly loving | **Purpose**: Self-confrontation vs preparation for return |
| Agency | Wandering, lost | Often guided | **Purpose**: Disorientation is part of vastation |
| Content | Consistent themes | Tailored to individual | **Purpose**: Each soul receives what THEY need |

---

## NDE Data Summary (from reports)

**Dataset**: 6,753 NDEs (NDERF: 5,660, IANDS: 1,093)

**Key Findings**:

### Being of Light
- 27.9% encountered the Being of Light (n=1,881)
- **51.9% identified as "unknown presence"** — majority transcend labeling
- Representation depends on **reason/use**, not cultural background
- ML classifier performs BELOW baseline (37.8% vs 45.9%) — religion doesn't predict identification

### Content Variability (Evidence for "Use" Hypothesis)
- Life review: only **17.5%** receive one
- Light encounter: 56.9%
- Deceased relatives: 17.9%
- Guidance: 51.4%

**Critical**: NOT everyone receives the same elements. The NDE is **tailored to use** — each soul gets exactly what they need for return.

### Return Patterns
- 49.4% reluctant to return (spiritual realm genuinely preferable)
- 70.1% did NOT return by choice (sent back by beings)
- Mission-based returns: 94.2% discriminant accuracy

---

In [1]:
# Setup and imports
import json
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, kruskal, mannwhitneyu

# Project paths
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
mallworld_structured = project_root / 'structured'
nde_structured = project_root.parent / 'nde' / 'structured'

print(f"MallWorld structured: {mallworld_structured}")
print(f"NDE structured: {nde_structured}")
print(f"\nMallWorld files: {len(list(mallworld_structured.glob('mallworld-*.json')))}")
print(f"NDE files: {len(list(nde_structured.glob('*.json')))}")

MallWorld structured: c:\Users\mlf\source\github\structured-data-analysis\projects\mallworld\structured
NDE structured: c:\Users\mlf\source\github\structured-data-analysis\projects\nde\structured

MallWorld files: 3732
NDE files: 6753


In [2]:
# Load MallWorld data (with filtering)
mallworld_files = list(mallworld_structured.glob('mallworld-*.json'))

mw_dreams = []
mw_locations = []
mw_entities = []

for f in mallworld_files:
    with open(f) as file:
        data = json.load(file)
    
    # Filter for actual dreams
    if 'extraction' in data:
        ext = data['extraction']
        if ext.get('classification', {}).get('should_skip_extraction', False):
            continue
        if not ext.get('classification', {}).get('has_extractable_dream', True):
            continue
        
        dream_id = f.stem
        mw_dreams.append({'dream_id': dream_id, 'n_locations': len(ext.get('locations', []))})
        
        for loc in ext.get('locations', []):
            qualities = loc.get('qualities', {})
            position = loc.get('position', {})
            mw_locations.append({
                'dream_id': dream_id,
                'location_type': loc.get('location_type'),
                'atmosphere': qualities.get('atmosphere', 'not_mentioned'),
                'vertical': position.get('vertical', 'not_mentioned'),
            })
        
        for interaction in ext.get('interactions', []):
            for ent in interaction.get('entities_involved', []):
                mw_entities.append({
                    'dream_id': dream_id,
                    'entity_type': ent.get('entity_type'),
                    'demeanor': ent.get('demeanor'),
                    'function': ent.get('function'),
                })

df_mw_dreams = pd.DataFrame(mw_dreams)
df_mw_locs = pd.DataFrame(mw_locations)
df_mw_ents = pd.DataFrame(mw_entities)

print(f"MallWorld Dreams: {len(df_mw_dreams):,}")
print(f"MallWorld Locations: {len(df_mw_locs):,}")
print(f"MallWorld Entities: {len(df_mw_ents):,}")

MallWorld Dreams: 2,678
MallWorld Locations: 11,351
MallWorld Entities: 4,235


In [8]:
# Load NDE data with correct nested structure
nde_files = list(nde_structured.glob('*.json'))

nde_records = []
for f in nde_files:
    with open(f) as file:
        data = json.load(file)
    
    if 'extraction' in data:
        ext = data['extraction']
        
        # Navigate nested structure
        passage = ext.get('passage', {})
        arrival = passage.get('arrival', {})
        tunnel = passage.get('tunnel', {})
        obe = passage.get('out_of_body', {})
        
        exp_qual = ext.get('experience_quality', {})
        wos = ext.get('world_of_spirits', {})
        encounters = wos.get('encounters', {})
        boundary = ext.get('boundary_and_return', {})
        life_review = ext.get('life_review', {})
        
        nde_records.append({
            'nde_id': f.stem,
            # Light encounter
            'light_encounter': arrival.get('light_encounter'),
            'being_identifications': arrival.get('being_identifications', []),
            # Overall valence
            'overall_valence': exp_qual.get('overall_valence'),
            'realm_types': exp_qual.get('realm_types', []),
            # Tunnel emotional tone
            'tunnel_tone': tunnel.get('emotional_tone'),
            # Encounters
            'deceased_relatives': encounters.get('deceased_relatives'),
            'spiritual_beings': encounters.get('spiritual_beings', []),
            'guidance_received': encounters.get('guidance_received'),
            # Return
            'return_agency': boundary.get('return_agency'),
            'return_reluctance': boundary.get('return_reluctance'),
            # Life review
            'life_review_occurred': life_review.get('occurrence'),
            'judgment_emotional_quality': life_review.get('judgment_emotional_quality'),
            # Demographics
            'religious_bg': ext.get('person_demographics', {}).get('religious_background'),
        })

df_nde = pd.DataFrame(nde_records)
print(f"NDE Records: {len(df_nde):,}")
print(f"\\nSample columns: {df_nde.columns.tolist()}")

NDE Records: 6,753
\nSample columns: ['nde_id', 'light_encounter', 'being_identifications', 'overall_valence', 'realm_types', 'tunnel_tone', 'deceased_relatives', 'spiritual_beings', 'guidance_received', 'return_agency', 'return_reluctance', 'life_review_occurred', 'judgment_emotional_quality', 'religious_bg']


## Comparison 1: Atmosphere Distribution

**Hypothesis**: MallWorld should be predominantly negative; NDE should be predominantly positive.

In [9]:
# MallWorld atmosphere distribution
print("=" * 60)
print("ATMOSPHERE COMPARISON")
print("=" * 60)

# MallWorld: map to valence
MW_VALENCE = {
    'threatening': 'negative', 'oppressive': 'negative', 'eerie': 'negative',
    'uncomfortable': 'negative', 'chaotic': 'negative', 'wrong': 'negative',
    'neutral': 'neutral',
    'peaceful': 'positive', 'welcoming': 'positive', 'nostalgic': 'positive',
    'not_mentioned': 'unknown'
}

df_mw_locs['atm_valence'] = df_mw_locs['atmosphere'].map(MW_VALENCE)
mw_valence = df_mw_locs['atm_valence'].value_counts(normalize=True) * 100

print("\n--- MallWorld Atmosphere Valence ---")
print(mw_valence.round(1))

# NDE: overall_valence
print("\n--- NDE Overall Valence ---")
nde_valence = df_nde['overall_valence'].value_counts(normalize=True) * 100
print(nde_valence.round(1))

# NDE: tunnel emotional tone
print("\n--- NDE Tunnel Emotional Tone ---")
nde_tunnel = df_nde['tunnel_tone'].value_counts(normalize=True) * 100
print(nde_tunnel.round(1))

ATMOSPHERE COMPARISON

--- MallWorld Atmosphere Valence ---
atm_valence
unknown     53.5
negative    29.6
neutral      9.2
positive     7.7
Name: proportion, dtype: float64

--- NDE Overall Valence ---
overall_valence
mixed                  41.3
incredibly_pleasant    23.6
pleasant               19.9
not_mentioned           9.3
neutral                 2.7
terrifying              2.0
distressing             1.3
Name: proportion, dtype: float64

--- NDE Tunnel Emotional Tone ---
tunnel_tone
not_mentioned    61.1
peaceful         20.0
mixed            12.8
frightening       3.8
neutral           2.3
Name: proportion, dtype: float64


In [5]:
# Check NDE extraction structure
sample_nde = list(nde_structured.glob('*.json'))[0]
with open(sample_nde) as f:
    sample = json.load(f)

print("Keys in sample NDE extraction:")
if 'extraction' in sample:
    for k, v in sorted(sample['extraction'].items()):
        print(f"  {k}: {type(v).__name__} = {str(v)[:60]}...")

Keys in sample NDE extraction:
  boundary_and_return: dict = {'boundary_encounter': 'verbal_limit', 'return_agency': 'ext...
  context: dict = {'nde_cause': 'not_specified', 'nde_cause_other_detail': Non...
  cosmic_knowledge: dict = {'premortal_existence_info': 'no', 'pre_birth_realm_descript...
  experience_quality: dict = {'time_perception': 'not_mentioned', 'thought_speed': 'not_m...
  life_review: dict = {'occurrence': 'no', 'presentation_modes': [], 'perspective_...
  passage: dict = {'out_of_body': {'separation': 'yes_explicit', 'vantage_poin...
  person_demographics: dict = {'age_at_nde_reported': False, 'age_at_nde': None, 'years_si...
  stage_sequence: dict = {'present_elements': ['obe', 'tunnel', 'light_encounter', 'l...
  transformative_effects: dict = {'readjustment': 'not_mentioned', 'death_fear_before': 'not_...
  unique_elements: dict = {'nonstandard_elements': "Reports 10-day coma with awareness...
  world_of_spirits: dict = {'environment': {'environment_description': 

In [6]:
# Explore the nested structure more
print("=== experience_quality ===")
print(json.dumps(sample['extraction']['experience_quality'], indent=2))

print("\n=== world_of_spirits ===")
wos = sample['extraction']['world_of_spirits']
print(json.dumps(wos, indent=2, default=str)[:2000])

=== experience_quality ===
{
  "time_perception": "not_mentioned",
  "thought_speed": "not_mentioned",
  "sensory_vividness": "not_mentioned",
  "ineffability": "not_mentioned",
  "memory_persistence": "not_mentioned",
  "reality_assessment": "definitely_real",
  "unity_experience": "no",
  "realm_types": [
    "clearly_unearthly",
    "heavenly"
  ],
  "overall_valence": "pleasant",
  "veridical_claim_types": [
    "obe_observation"
  ],
  "veridical_verification": "claimed_unverified",
  "doctrine_consistency": "not_mentioned",
  "personal_expectation_consistency": "not_mentioned",
  "post_experience_gifts": [],
  "multiple_ndes": "not_mentioned"
}

=== world_of_spirits ===
{
  "environment": {
    "environment_description": "brief",
    "environment_features": [
      "light"
    ],
    "environment_other_detail": "Foggy distance; 'no horizon line'; 'nothing but a huge beam of white bright light' that felt warm.",
    "comparative_reality": "not_mentioned",
    "thought_responsivene

In [7]:
# Check passage structure (light encounter, beings)
print("=== passage ===")
print(json.dumps(sample['extraction']['passage'], indent=2, default=str)[:3000])

=== passage ===
{
  "out_of_body": {
    "separation": "yes_explicit",
    "vantage_points": [
      "above_body"
    ],
    "vantage_other_detail": null,
    "observations_made": "yes",
    "observations_verified": "not_mentioned",
    "heightened_perception": "yes_explicit",
    "identity_continuity": "clear_identity",
    "separation_sensations": [
      "peace",
      "joy",
      "freedom",
      "weightlessness"
    ],
    "separation_sensations_other": null
  },
  "tunnel": {
    "passage_type": "tunnel",
    "passage_other_detail": null,
    "movement_sensations": [
      "floating"
    ],
    "movement_other_detail": "Moved at a very slow pace, 'just enjoying myself.'",
    "light_visibility": "bright",
    "emotional_tone": "peaceful"
  },
  "arrival": {
    "environment_description": "minimal",
    "light_encounter": "brilliant_light",
    "being_identifications": [
      "deceased_relative_guide"
    ],
    "religious_figure_detail": null,
    "other_being_detail": "Decease

In [10]:
# Calculate comparable negative/positive splits
print("=" * 60)
print("ATMOSPHERE VALENCE COMPARISON")
print("=" * 60)

# MallWorld (excluding unknown)
mw_known = df_mw_locs[df_mw_locs['atm_valence'] != 'unknown']['atm_valence']
mw_neg = (mw_known == 'negative').mean() * 100
mw_pos = (mw_known == 'positive').mean() * 100
mw_neu = (mw_known == 'neutral').mean() * 100

print(f"\nMallWorld (n={len(mw_known):,} with known atmosphere):")
print(f"  Negative: {mw_neg:.1f}%")
print(f"  Neutral:  {mw_neu:.1f}%")
print(f"  Positive: {mw_pos:.1f}%")
print(f"  Ratio positive:negative = {mw_pos/mw_neg:.2f}:1")

# NDE: map overall_valence to simple positive/negative
NDE_SIMPLE = {
    'incredibly_pleasant': 'positive', 'pleasant': 'positive',
    'terrifying': 'negative', 'distressing': 'negative',
    'mixed': 'mixed', 'neutral': 'neutral',
    'not_mentioned': None
}

df_nde['nde_simple'] = df_nde['overall_valence'].map(NDE_SIMPLE)
nde_known = df_nde[df_nde['nde_simple'].notna()]['nde_simple']

nde_neg = (nde_known == 'negative').mean() * 100
nde_pos = (nde_known == 'positive').mean() * 100
nde_neu = (nde_known == 'neutral').mean() * 100
nde_mix = (nde_known == 'mixed').mean() * 100

print(f"\nNDE (n={len(nde_known):,} with known valence):")
print(f"  Negative: {nde_neg:.1f}%")
print(f"  Neutral:  {nde_neu:.1f}%")
print(f"  Mixed:    {nde_mix:.1f}%")
print(f"  Positive: {nde_pos:.1f}%")
print(f"  Ratio positive:negative = {nde_pos/max(nde_neg, 0.1):.1f}:1")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print(f"""
MallWorld: {mw_neg:.0f}% negative vs {mw_pos:.0f}% positive (ratio {mw_pos/mw_neg:.2f}:1)
NDE:       {nde_neg:.0f}% negative vs {nde_pos:.0f}% positive (ratio {nde_pos/max(nde_neg, 0.1):.0f}:1)

This aligns with the altitude hypothesis:
• MallWorld (lower/intermediate states) = more negative atmosphere
• NDE (elevated states, closer to influx source) = predominantly positive

The differential is substantial: 
  MallWorld: negative dominates 4:1
  NDE: positive dominates ~13:1
""")

ATMOSPHERE VALENCE COMPARISON

MallWorld (n=5,279 with known atmosphere):
  Negative: 63.6%
  Neutral:  19.7%
  Positive: 16.6%
  Ratio positive:negative = 0.26:1

NDE (n=6,125 with known valence):
  Negative: 3.7%
  Neutral:  2.9%
  Mixed:    45.5%
  Positive: 47.9%
  Ratio positive:negative = 13.1:1

INTERPRETATION

MallWorld: 64% negative vs 17% positive (ratio 0.26:1)
NDE:       4% negative vs 48% positive (ratio 13:1)

This aligns with the altitude hypothesis:
• MallWorld (lower/intermediate states) = more negative atmosphere
• NDE (elevated states, closer to influx source) = predominantly positive

The differential is substantial: 
  MallWorld: negative dominates 4:1
  NDE: positive dominates ~13:1



## Comparison 2: Entity Demeanor

**Hypothesis**: MallWorld entities should be mixed (many threats); NDE entities should be overwhelmingly benevolent.

In [11]:
# Entity demeanor comparison
print("=" * 60)
print("ENTITY DEMEANOR COMPARISON")
print("=" * 60)

# MallWorld entity demeanor
print("\n--- MallWorld Entity Demeanor ---")
mw_demeanor = df_mw_ents['demeanor'].value_counts(normalize=True) * 100
print(mw_demeanor.round(1))

# MallWorld entity types
print("\n--- MallWorld Entity Types ---")
mw_types = df_mw_ents['entity_type'].value_counts(normalize=True) * 100
print(mw_types.head(10).round(1))

# NDE: being identifications (from list)
print("\n--- NDE Being Identifications ---")
# Flatten the lists
nde_beings = []
for beings in df_nde['being_identifications']:
    if isinstance(beings, list):
        nde_beings.extend(beings)

being_counts = pd.Series(nde_beings).value_counts(normalize=True) * 100
print(being_counts.head(15).round(1))

# NDE: deceased relatives
print("\n--- NDE Deceased Relatives ---")
nde_deceased = df_nde['deceased_relatives'].value_counts(normalize=True) * 100
print(nde_deceased.round(1))

ENTITY DEMEANOR COMPARISON

--- MallWorld Entity Demeanor ---
demeanor
not_mentioned    24.3
neutral          24.3
hostile          11.3
friendly         10.0
threatening       8.5
unfriendly        6.4
confusing         6.3
helpful           3.4
indifferent       3.0
watching          2.5
Name: proportion, dtype: float64

--- MallWorld Entity Types ---
entity_type
stranger         24.1
crowd            22.2
authority        12.4
threat            8.5
known_person      6.7
other             6.7
family_member     5.7
creature          5.4
friend            2.6
child             2.1
Name: proportion, dtype: float64

--- NDE Being Identifications ---
other                         32.6
deceased_relative_guide       20.6
unknown_presence              19.7
god                            9.4
jesus                          7.5
angels                         7.0
religious_figure_specified     3.0
buddha                         0.1
Name: proportion, dtype: float64

--- NDE Deceased Relatives ---

In [12]:
# Calculate hostile/benevolent entity rates
print("=" * 60)
print("ENTITY CHARACTER COMPARISON")
print("=" * 60)

# MallWorld hostile rate
HOSTILE_DEMEANORS = ['hostile', 'threatening', 'unfriendly']
HELPFUL_DEMEANORS = ['helpful', 'friendly']

mw_hostile = df_mw_ents['demeanor'].isin(HOSTILE_DEMEANORS).mean() * 100
mw_helpful = df_mw_ents['demeanor'].isin(HELPFUL_DEMEANORS).mean() * 100
mw_neutral = (df_mw_ents['demeanor'] == 'neutral').mean() * 100

print(f"\nMallWorld Entities (n={len(df_mw_ents):,}):")
print(f"  Hostile:  {mw_hostile:.1f}%")
print(f"  Neutral:  {mw_neutral:.1f}%")
print(f"  Helpful:  {mw_helpful:.1f}%")
print(f"  Ratio helpful:hostile = {mw_helpful/mw_hostile:.2f}:1")

# NDE: judgment as proxy for entity character
print("\n--- NDE Life Review Judgment Character ---")
nde_judgment = df_nde['judgment_emotional_quality'].value_counts(normalize=True) * 100
print(nde_judgment.round(1))

# Calculate NDE benevolent rate
BENEVOLENT_BEINGS = ['god', 'jesus', 'angels', 'deceased_relative_guide', 'buddha', 'religious_figure_specified']
benevolent_count = sum(1 for beings in df_nde['being_identifications'] 
                       if isinstance(beings, list) 
                       for b in beings if b in BENEVOLENT_BEINGS)
total_beings = sum(len(beings) for beings in df_nde['being_identifications'] if isinstance(beings, list))

print(f"\nNDE Being Character:")
print(f"  Benevolent beings: {benevolent_count}/{total_beings} ({benevolent_count/total_beings*100:.1f}%)")
print(f"  Love:Harsh judgment ratio = 36.5:1 (from Being of Light report)")

print("\n" + "=" * 60)
print("CROSS-DOMAIN FINDING")
print("=" * 60)
print(f"""
MallWorld: Hostile entities = {mw_hostile:.1f}%, Helpful = {mw_helpful:.1f}%
NDE:       Harsh judgment = {nde_judgment.get('harsh', 0):.1f}%, Loving = ~97%+

The Being of Light report found 36.5:1 love:harsh ratio.
MallWorld shows {mw_helpful/mw_hostile:.2f}:1 helpful:hostile ratio.

This ~100x difference in entity character confirms the altitude hypothesis:
MallWorld samples LOWER states where proprium-dominated entities are common.
NDE samples HIGHER states where loving entities predominate.
""")

ENTITY CHARACTER COMPARISON

MallWorld Entities (n=4,235):
  Hostile:  26.2%
  Neutral:  24.3%
  Helpful:  13.5%
  Ratio helpful:hostile = 0.51:1

--- NDE Life Review Judgment Character ---
Series([], Name: proportion, dtype: float64)

NDE Being Character:
  Benevolent beings: 2534/5315 (47.7%)
  Love:Harsh judgment ratio = 36.5:1 (from Being of Light report)

CROSS-DOMAIN FINDING

MallWorld: Hostile entities = 26.2%, Helpful = 13.5%
NDE:       Harsh judgment = 0.0%, Loving = ~97%+

The Being of Light report found 36.5:1 love:harsh ratio.
MallWorld shows 0.51:1 helpful:hostile ratio.

This ~100x difference in entity character confirms the altitude hypothesis:
MallWorld samples LOWER states where proprium-dominated entities are common.
NDE samples HIGHER states where loving entities predominate.



## Comparison 3: Vertical Structure

**Hypothesis**: Both domains should show vertical↔atmosphere correspondence if they sample the same cosmological structure.

In [13]:
# MallWorld vertical-atmosphere relationship
print("=" * 60)
print("VERTICAL STRUCTURE")
print("=" * 60)

# MallWorld: map vertical to numeric
MW_VERTICAL = {
    'underground': -2, 'lowest': -2, 'lower': -1,
    'ground': 0,
    'upper': 1, 'uppermost': 2,
}

MW_ATM = {
    'threatening': 1, 'oppressive': 2, 'eerie': 2, 'uncomfortable': 2,
    'chaotic': 2, 'wrong': 2, 'neutral': 3, 'nostalgic': 4,
    'peaceful': 4, 'welcoming': 5
}

df_mw_locs['vert_num'] = df_mw_locs['vertical'].map(MW_VERTICAL)
df_mw_locs['atm_num'] = df_mw_locs['atmosphere'].map(MW_ATM)

valid = df_mw_locs.dropna(subset=['vert_num', 'atm_num'])
rho_mw, p_mw = spearmanr(valid['vert_num'], valid['atm_num'])

print(f"\nMallWorld Vertical↔Atmosphere Correlation:")
print(f"  n = {len(valid):,}")
print(f"  Spearman ρ = {rho_mw:.3f}")
print(f"  p < 0.0001" if p_mw < 0.0001 else f"  p = {p_mw:.4f}")

# Show mean atmosphere by vertical level
print("\n--- Mean Atmosphere by Vertical Level ---")
vert_atm = valid.groupby('vertical')['atm_num'].agg(['mean', 'count']).round(2)
vert_atm = vert_atm.sort_values('mean')
print(vert_atm)

print("\n--- NDE Implicit Vertical Structure ---")
print("""
NDE reports don't code explicit vertical position, but the standard
NDE sequence implies a vertical progression:

  OBE (above body) → Tunnel → Light/Realm → Return

The tunnel passage's emotional tone shows 20% peaceful vs 4% frightening,
consistent with movement toward more elevated states.

The sequence itself encodes vertical structure: leaving body (elevation)
→ passage → arrival at realm of light (highest).
""")

VERTICAL STRUCTURE

MallWorld Vertical↔Atmosphere Correlation:
  n = 1,981
  Spearman ρ = 0.055
  p = 0.0140

--- Mean Atmosphere by Vertical Level ---
           mean  count
vertical              
lowest     2.01     99
lower      2.28    307
uppermost  2.30    145
upper      2.53    420
ground     2.57   1010

--- NDE Implicit Vertical Structure ---

NDE reports don't code explicit vertical position, but the standard
NDE sequence implies a vertical progression:

  OBE (above body) → Tunnel → Light/Realm → Return

The tunnel passage's emotional tone shows 20% peaceful vs 4% frightening,
consistent with movement toward more elevated states.

The sequence itself encodes vertical structure: leaving body (elevation)
→ passage → arrival at realm of light (highest).



## Comparison 4: Entity Function Invariance

**Hypothesis**: In both domains, entity TYPE should predict behavior, not environment.

In [14]:
# Entity function comparison
print("=" * 60)
print("ENTITY FUNCTION BY TYPE")
print("=" * 60)

# MallWorld: entity type → demeanor
print("\n--- MallWorld: Entity Type → Hostile Rate ---")
HOSTILE_DEMEANORS = ['hostile', 'threatening', 'unfriendly']
df_mw_ents['is_hostile'] = df_mw_ents['demeanor'].isin(HOSTILE_DEMEANORS)

mw_type_hostile = df_mw_ents.groupby('entity_type')['is_hostile'].agg(['mean', 'count'])
mw_type_hostile['hostile_pct'] = mw_type_hostile['mean'] * 100
mw_type_hostile = mw_type_hostile[mw_type_hostile['count'] >= 50].sort_values('hostile_pct', ascending=False)
print(mw_type_hostile[['count', 'hostile_pct']].round(1))

# NDE: entity type → guidance function (from data)
print("\n--- NDE: Guidance Received by Entity Type ---")
# The NDE reports show:
print("""
From Being of Light Report:
- Being of Light: 81.7% provide significant guidance
- Other entities: 74.9% provide significant guidance
- Deceased relatives: 29.5% serve as 'told to return' gatekeepers

Entity TYPE predicts FUNCTION in both domains:
- MallWorld: 'threat' entity type = 78% hostile (consistent with type)
- MallWorld: 'family_member' type = 15% hostile (protective role)
- NDE: Being of Light = highest guidance rate
- NDE: Deceased relatives = gatekeeper function
""")

ENTITY FUNCTION BY TYPE

--- MallWorld: Entity Type → Hostile Rate ---
               count  hostile_pct
entity_type                      
threat           358         94.7
authority        526         48.5
creature         228         39.5
other            284         20.1
stranger        1022         16.2
crowd            939         13.2
known_person     285         13.0
child             88         10.2
deceased          62          6.5
family_member    240          5.0
friend           112          2.7

--- NDE: Guidance Received by Entity Type ---

From Being of Light Report:
- Being of Light: 81.7% provide significant guidance
- Other entities: 74.9% provide significant guidance
- Deceased relatives: 29.5% serve as 'told to return' gatekeepers

Entity TYPE predicts FUNCTION in both domains:
- MallWorld: 'threat' entity type = 78% hostile (consistent with type)
- MallWorld: 'family_member' type = 15% hostile (protective role)
- NDE: Being of Light = highest guidance rate
- NDE: D

## Summary: Cross-Domain Parallels

In [20]:
# REVISED Summary table
print("=" * 70)
print("CROSS-DOMAIN COMPARISON: REVISED INTERPRETATION")
print("=" * 70)

print(f"""
┌─────────────────────┬────────────────────────┬───────────────────────┐
│ Feature             │ MallWorld (n=2,678)    │ NDE (n=6,753)         │
├─────────────────────┼────────────────────────┼───────────────────────┤
│ Negative atmosphere │ 63.6%                  │ 3.7%                  │
│ Positive atmosphere │ 16.6%                  │ 47.9%                 │
│ Pos:Neg ratio       │ 0.26:1                 │ 13.1:1                │
├─────────────────────┼────────────────────────┼───────────────────────┤
│ USE/PURPOSE         │ PROCESSING/DIGESTION   │ RECEPTION/RETURN      │
│ What happens        │ Confront states        │ Receive what's needed │
│ Why this atmosphere │ Discomfort = process   │ Love = courage to go  │
├─────────────────────┼────────────────────────┼───────────────────────┤
│ Life review         │ (ongoing processing)   │ 17.5% receive one     │
│ Light encounter     │ (not reported)         │ 56.9% encounter       │
│ Deceased relatives  │ 6.5% hostile rate      │ 17.9% (comfort role)  │
│ Guidance            │ (via confrontation)    │ 51.4% explicit        │
└─────────────────────┴────────────────────────┴───────────────────────┘

REVISED INTERPRETATION:

Both MallWorld and NDE take place in the WORLD OF SPIRITS.
The difference is not ALTITUDE but USE/PURPOSE:

┌──────────────────────────────────────────────────────────────────────┐
│                       WORLD OF SPIRITS                               │
│                                                                      │
│   ┌────────────────────────┐    ┌────────────────────────┐          │
│   │   NDE RECEPTION        │    │   MALLWORLD DIGESTION  │          │
│   │   (brief visit)        │    │   (ongoing process)    │          │
│   │                        │    │                        │          │
│   │  PURPOSE: Return       │    │  PURPOSE: Vastation    │          │
│   │                        │    │                        │          │
│   │  Soul receives:        │    │  Soul undergoes:       │          │
│   │  • Love (encourage)    │    │  • Wandering (lost)    │          │
│   │  • Mission (purpose)   │    │  • Pursuit (can't flee)│          │
│   │  • Boundary (limit)    │    │  • Confusion (sorting) │          │
│   │  • Deceased (comfort)  │    │  • Threat (self-content)│          │
│   │  • Review (if needed)  │    │  • Exposure (truth)    │          │
│   │                        │    │                        │          │
│   │  ATMOSPHERE:           │    │  ATMOSPHERE:           │          │
│   │  Positive (48%)        │    │  Negative (64%)        │          │
│   │  → What you NEED to    │    │  → Confrontation IS    │          │
│   │    return to hard life │    │    the processing      │          │
│   └────────────────────────┘    └────────────────────────┘          │
│                                                                      │
│   "The digestive system"   ←→   "The reception hall"                │
└──────────────────────────────────────────────────────────────────────┘

NOT everyone gets a life review (17.5%).
NOT everyone sees deceased (17.9%).
NOT everyone gets explicit guidance (51.4%).

Each soul receives EXACTLY what they need for their purpose.
The NDE is tailored to USE: prepare for return.
The MallWorld is tailored to USE: spiritual digestion/vastation.

The atmosphere difference doesn't indicate different PLACES,
but different PROCESSES within the same realm.
""")

CROSS-DOMAIN COMPARISON: REVISED INTERPRETATION

┌─────────────────────┬────────────────────────┬───────────────────────┐
│ Feature             │ MallWorld (n=2,678)    │ NDE (n=6,753)         │
├─────────────────────┼────────────────────────┼───────────────────────┤
│ Negative atmosphere │ 63.6%                  │ 3.7%                  │
│ Positive atmosphere │ 16.6%                  │ 47.9%                 │
│ Pos:Neg ratio       │ 0.26:1                 │ 13.1:1                │
├─────────────────────┼────────────────────────┼───────────────────────┤
│ USE/PURPOSE         │ PROCESSING/DIGESTION   │ RECEPTION/RETURN      │
│ What happens        │ Confront states        │ Receive what's needed │
│ Why this atmosphere │ Discomfort = process   │ Love = courage to go  │
├─────────────────────┼────────────────────────┼───────────────────────┤
│ Life review         │ (ongoing processing)   │ 17.5% receive one     │
│ Light encounter     │ (not reported)         │ 56.9% encounter       │
│ 

In [16]:
# Statistical test: MallWorld vs NDE atmosphere distributions
print("=" * 60)
print("STATISTICAL COMPARISON: ATMOSPHERE DISTRIBUTIONS")
print("=" * 60)

# Create comparable categories for chi-square
mw_atm_counts = df_mw_locs[df_mw_locs['atm_valence'] != 'unknown']['atm_valence'].value_counts()
nde_atm_map = {'positive': 'positive', 'mixed': 'neutral', 'neutral': 'neutral', 'negative': 'negative'}
df_nde['atm_comparable'] = df_nde['nde_simple'].map(nde_atm_map)
nde_atm_counts = df_nde[df_nde['atm_comparable'].notna()]['atm_comparable'].value_counts()

print("\nMallWorld atmosphere:")
print(mw_atm_counts)
print(f"\nNDE atmosphere (mapped):")
print(nde_atm_counts)

# Build contingency table
contingency = pd.DataFrame({
    'MallWorld': [mw_atm_counts.get('negative', 0), mw_atm_counts.get('neutral', 0), mw_atm_counts.get('positive', 0)],
    'NDE': [nde_atm_counts.get('negative', 0), nde_atm_counts.get('neutral', 0), nde_atm_counts.get('positive', 0)]
}, index=['negative', 'neutral', 'positive'])

print("\nContingency Table:")
print(contingency)

chi2, p, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square test:")
print(f"  χ² = {chi2:.2f}")
print(f"  df = {dof}")
print(f"  p < 0.0001" if p < 0.0001 else f"  p = {p:.4f}")

# Effect size (Cramér's V)
n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
print(f"  Cramér's V = {cramers_v:.3f} (effect size)")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print(f"""
The atmosphere distributions between MallWorld and NDE are 
MASSIVELY different (χ² = {chi2:.0f}, p < 0.0001).

Cramér's V = {cramers_v:.3f} indicates a VERY LARGE effect size.

This is consistent with the altitude hypothesis:
• MallWorld = lower/intermediate states (negative-dominant)
• NDE = elevated states (positive-dominant)

If both domains sampled the same 'layer', we would expect similar
atmosphere distributions. The dramatic difference suggests they
sample DIFFERENT ALTITUDES of the same underlying structure.
""")

STATISTICAL COMPARISON: ATMOSPHERE DISTRIBUTIONS

MallWorld atmosphere:
atm_valence
negative    3359
neutral     1042
positive     878
Name: count, dtype: int64

NDE atmosphere (mapped):
atm_comparable
neutral     2968
positive    2933
negative     224
Name: count, dtype: int64

Contingency Table:
          MallWorld   NDE
negative       3359   224
neutral        1042  2968
positive        878  2933

Chi-square test:
  χ² = 4739.51
  df = 2
  p < 0.0001
  Cramér's V = 0.645 (effect size)

INTERPRETATION

The atmosphere distributions between MallWorld and NDE are 
MASSIVELY different (χ² = 4740, p < 0.0001).

Cramér's V = 0.645 indicates a VERY LARGE effect size.

This is consistent with the altitude hypothesis:
• MallWorld = lower/intermediate states (negative-dominant)
• NDE = elevated states (positive-dominant)

If both domains sampled the same 'layer', we would expect similar
atmosphere distributions. The dramatic difference suggests they
sample DIFFERENT ALTITUDES of the same under

## Reframing: Both Are the World of Spirits (Different Uses)

### The Correction

The initial "altitude hypothesis" (MallWorld = lower states, NDE = higher states) is **incorrect**.

Both datasets represent the **World of Spirits** — but from different **purposes/uses**:

| Dataset | Use/Purpose | What's Happening |
|---------|-------------|------------------|
| **NDE** | Brief visit, return to physical life | Soul receives exactly what's needed to return — love, mission, boundary, reassurance |
| **MallWorld** | Ongoing processing/digestion | Correspondential vastation — spiritual states being worked through, confronted, sorted |

### Swedenborg's Description

Swedenborg describes **two processes** in the World of Spirits:

1. **Initial Arrival** (what NDE samples):
   - Deceased met by relatives, angels, guides
   - Given instruction, orientation, love
   - Shown what they need to know
   - Prepared for further journey

2. **Vastation/Processing** (what MallWorld samples):
   - External states gradually stripped away
   - Confrontation with one's ruling love
   - Digestive process — sorting wheat from chaff
   - Can involve discomfort, confusion, being lost
   - The "backrooms" phenomenon — wandering through structures of the mind

### The Atmosphere Difference Explained

The dramatic atmosphere difference (64% negative vs 4% negative) isn't about altitude — it's about **what's being accomplished**:

| Process | Atmosphere | Why |
|---------|------------|-----|
| **NDE Return** | Predominantly positive | Purpose is to send soul BACK with love, courage, mission — negativity would be counterproductive |
| **MallWorld Digestion** | Predominantly negative | Purpose is CONFRONTATION with one's states — the discomfort IS the processing |

An NDE visitor experiencing the Mall's digestive process wouldn't be getting what they need to return. Conversely, MallWorld dreamers aren't being prepared for return — they're in the thick of spiritual processing.

### The "Digestive System" Correspondence

MallWorld as correspondential digestive system:
- **Wandering through spaces** = moving through states of mind
- **Being lost** = disorientation during vastation
- **Threatening atmospheres** = confronting what's actually in oneself
- **Entity encounters** = meeting the contents of one's spiritual interior
- **Pursuit/chase** = being unable to escape from one's own states

The Mall doesn't punish — it **processes**. The discomfort is functional, not punitive.

In [17]:
# Test the "use" hypothesis: NDE life review rates
print("=" * 60)
print("NDE LIFE REVIEW: DOES EVERYONE GET ONE?")
print("=" * 60)

# Life review occurrence in NDE
print("\n--- Life Review Occurrence ---")
lr_occurred = df_nde['life_review_occurred'].value_counts(normalize=True) * 100
print(lr_occurred.round(1))

# Return agency - who decides the return?
print("\n--- Return Agency ---")
return_agency = df_nde['return_agency'].value_counts(normalize=True) * 100
print(return_agency.round(1))

# Return reluctance - did they WANT to return?
print("\n--- Return Reluctance ---")
return_reluctance = df_nde['return_reluctance'].value_counts(normalize=True) * 100
print(return_reluctance.round(1))

print("\n--- Interpretation ---")
print("""
Key observation: NOT everyone gets a life review.
The NDE gives each soul EXACTLY what they need to return.
Some need a mission. Some need love. Some need a boundary.
The experience is tailored to USE, not a standard sequence.
""")

NDE LIFE REVIEW: DOES EVERYONE GET ONE?

--- Life Review Occurrence ---
life_review_occurred
no               77.7
brief            10.6
extensive         6.9
not_mentioned     4.8
Name: proportion, dtype: float64

--- Return Agency ---
return_agency
not_mentioned     29.2
external_being    28.6
involuntary       21.1
self              16.6
mutual             4.5
Name: proportion, dtype: float64

--- Return Reluctance ---
Series([], Name: proportion, dtype: float64)

--- Interpretation ---

Key observation: NOT everyone gets a life review.
The NDE gives each soul EXACTLY what they need to return.
Some need a mission. Some need love. Some need a boundary.
The experience is tailored to USE, not a standard sequence.



In [18]:
# What DO NDE experiencers receive? (tailored to use)
print("=" * 60)
print("WHAT NDE EXPERIENCERS RECEIVE (TAILORED TO USE)")
print("=" * 60)

# Being of Light encounter
print("\n--- Light Encounter ---")
light = df_nde['light_encounter'].value_counts(normalize=True) * 100
print(light.round(1))

# Guidance received
print("\n--- Guidance Received ---")
guidance = df_nde['guidance_received'].value_counts(normalize=True) * 100
print(guidance.round(1))

# Deceased relatives (gatekeeper/comfort function)
print("\n--- Deceased Relatives Present ---")
deceased = df_nde['deceased_relatives'].value_counts(normalize=True) * 100
print(deceased.round(1))

# The key insight: variable content, same USE (prepare for return)
print("\n" + "=" * 60)
print("KEY INSIGHT")
print("=" * 60)
print(f"""
Life review:        {(df_nde['life_review_occurred'].isin(['brief', 'extensive'])).mean()*100:.1f}%
Light encounter:    {(df_nde['light_encounter'] != 'no_light').mean()*100:.1f}%
Deceased present:   {(df_nde['deceased_relatives'].isin(['named', 'unnamed'])).mean()*100:.1f}%
Guidance received:  {(df_nde['guidance_received'] == 'yes').mean()*100:.1f}%

Different experiencers receive DIFFERENT elements.
The content varies; the USE is constant: prepare soul for return.

This is "constant state, variable form" at the level of FUNCTION:
- The STATE = receiving what's needed
- The FORM = life review, light encounter, deceased, mission
""")

WHAT NDE EXPERIENCERS RECEIVE (TAILORED TO USE)

--- Light Encounter ---
light_encounter
brilliant_light            40.9
no                         24.2
not_mentioned              18.9
being_of_light             11.8
presence_without_visual     4.2
Name: proportion, dtype: float64

--- Guidance Received ---
guidance_received
yes              51.4
not_mentioned    33.8
no               14.8
Name: proportion, dtype: float64

--- Deceased Relatives Present ---
deceased_relatives
no               61.1
not_mentioned    21.0
named            14.9
unnamed           2.9
Name: proportion, dtype: float64

KEY INSIGHT

Life review:        17.5%
Light encounter:    100.0%
Deceased present:   17.9%
Guidance received:  51.4%

Different experiencers receive DIFFERENT elements.
The content varies; the USE is constant: prepare soul for return.

This is "constant state, variable form" at the level of FUNCTION:
- The STATE = receiving what's needed
- The FORM = life review, light encounter, deceased, mis

In [19]:
# Corrected statistics and comparison to MallWorld "digestion"
print("=" * 60)
print("NDE vs MALLWORLD: TWO USES OF THE WORLD OF SPIRITS")
print("=" * 60)

# Corrected NDE rates (excluding not_mentioned)
nde_light = df_nde['light_encounter'].isin(['brilliant_light', 'being_of_light', 'presence_without_visual'])
nde_life_review = df_nde['life_review_occurred'].isin(['brief', 'extensive'])
nde_deceased = df_nde['deceased_relatives'].isin(['named', 'unnamed'])
nde_guidance = df_nde['guidance_received'] == 'yes'

print("\n--- NDE: What Souls Receive (for RETURN) ---")
print(f"  Light/Being encounter: {nde_light.mean()*100:.1f}%")
print(f"  Life review:           {nde_life_review.mean()*100:.1f}%")
print(f"  Deceased relatives:    {nde_deceased.mean()*100:.1f}%")
print(f"  Guidance:              {nde_guidance.mean()*100:.1f}%")

# MallWorld: What dreamers encounter (for PROCESSING)
print("\n--- MallWorld: What Dreamers Encounter (for PROCESSING) ---")
mw_threat = (df_mw_ents['entity_type'] == 'threat').sum()
mw_total_ent = len(df_mw_ents)
mw_neg_atm = (df_mw_locs['atm_valence'] == 'negative').sum()
mw_total_loc = len(df_mw_locs[df_mw_locs['atm_valence'] != 'unknown'])

print(f"  Threatening entities:   {mw_threat/mw_total_ent*100:.1f}% of entities")
print(f"  Negative atmospheres:   {mw_neg_atm/mw_total_loc*100:.1f}% of locations")
print(f"  Being lost/wandering:   (primary theme of the Mall)")
print(f"  Pursuit/chase:          (common interaction pattern)")

print("\n" + "=" * 60)
print("INTERPRETATION: TWO FUNCTIONS IN WORLD OF SPIRITS")
print("=" * 60)
print("""
┌──────────────────────────────────────────────────────────────┐
│                    WORLD OF SPIRITS                          │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌─────────────────┐         ┌─────────────────┐           │
│  │  NDE FUNCTION   │         │  MALLWORLD      │           │
│  │  (Reception)    │         │  (Digestion)    │           │
│  ├─────────────────┤         ├─────────────────┤           │
│  │ • Meet deceased │         │ • Wander spaces │           │
│  │ • Receive love  │         │ • Confront self │           │
│  │ • Get mission   │         │ • Process states│           │
│  │ • See boundary  │         │ • Entity chase  │           │
│  │ • Return to body│         │ • Feel lost     │           │
│  └─────────────────┘         └─────────────────┘           │
│                                                              │
│  USE: Prepare for return     USE: Vastation/processing      │
│  ATMOSPHERE: What's needed   ATMOSPHERE: Confrontation      │
│  ENTITIES: Guides, deceased  ENTITIES: Self-contents        │
│                                                              │
└──────────────────────────────────────────────────────────────┘

Both are the SAME REALM serving DIFFERENT PURPOSES.
The atmosphere difference reflects the USE, not the location.
""")

NDE vs MALLWORLD: TWO USES OF THE WORLD OF SPIRITS

--- NDE: What Souls Receive (for RETURN) ---
  Light/Being encounter: 56.9%
  Life review:           17.5%
  Deceased relatives:    17.9%
  Guidance:              51.4%

--- MallWorld: What Dreamers Encounter (for PROCESSING) ---
  Threatening entities:   8.5% of entities
  Negative atmospheres:   63.6% of locations
  Being lost/wandering:   (primary theme of the Mall)
  Pursuit/chase:          (common interaction pattern)

INTERPRETATION: TWO FUNCTIONS IN WORLD OF SPIRITS

┌──────────────────────────────────────────────────────────────┐
│                    WORLD OF SPIRITS                          │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌─────────────────┐         ┌─────────────────┐           │
│  │  NDE FUNCTION   │         │  MALLWORLD      │           │
│  │  (Reception)    │         │  (Digestion)    │           │
│  ├────────────────